Run saved greeting audio clips directly through the deployed OpenVINO model and inspect its output. The model receives the Subway hotword and Thai/English language instructions, with no timestamp instructions added. The model built-in transcription prompt remains required by its chat audio interface. Original transcripts are used only to select test clips and are not passed to the model or compared against its output. Run cells with the project-local `.venv` kernel. Every clip uses a fresh session that is unloaded in `finally`.



In [1]:
from pathlib import Path
import base64
from dataclasses import asdict
from datetime import datetime, timezone
from html import escape
import json
import os
import sys
from time import perf_counter
import unicodedata

from IPython.display import HTML, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "speech_to_text_deploy/settings.json").is_file())
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select the project-local .venv kernel."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from speech_to_text_deploy.config import load_settings, new_run_dir
from speech_to_text_deploy.runtime import openvino_environment, release_model_memory, tensor_memory

environment = openvino_environment()
if "LD_LIBRARY_PATH" in environment:
    os.environ["LD_LIBRARY_PATH"] = environment["LD_LIBRARY_PATH"]
else:
    os.environ.pop("LD_LIBRARY_PATH", None)

import soundfile as sf
import torch
from speech_to_text_deploy.model.session import OpenVinoSession

torch.set_num_threads(load_settings()["cpu_threads"])
settings = load_settings()
# Add the requested hotword and language guidance.
settings["hotwords"] = ("Subway","ซัพเวย์","ซับเวย์")
settings["languages"] = ("Thai", "English")
MAX_NEW_TOKENS = min(settings["max_new_tokens"], 128)
manifest = json.loads((Path(settings["model_dir"]) / "manifest.json").read_text())
assert manifest["model_id"] == settings["model_id"] == "faiisu/moss-asr-th-finetuned-subway"
print("Model:", settings["model_id"])
print("Snapshot:", manifest["source_snapshot"])
print("Device:", settings["ov_device"])







Model: faiisu/moss-asr-th-finetuned-subway
Snapshot: 736dc9e54eb1d2e06123af4b6f24f36c84a6609c
Device: GPU


## Input audio
Select every matching saved utterance in chronological order and keep its original transcript and recording offset.

In [2]:
SOURCE_RUN = ROOT / "research_output_gathering/output/run-20261102T025949Z-b3cdc132"
KEYWORD = "สวัสดี"
inputs = []
for metadata_path in sorted((SOURCE_RUN / "results").glob("segment-*/utterance-*.json")):
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if KEYWORD not in metadata.get("text", ""):
        continue
    audio_path = metadata_path.with_suffix(".wav")
    if not audio_path.is_file():
        raise FileNotFoundError(audio_path)
    audio_info = sf.info(audio_path)
    inputs.append({
        "id": f"{metadata_path.parent.name}/{audio_path.stem}",
        "audio_path": str(audio_path),
        "metadata_path": str(metadata_path),
        "original_text": metadata["text"],
        "original_speaker": metadata["speaker"],
        "recording_start_seconds": metadata["recording_start_seconds"],
        "start_utc": metadata.get("start_utc"),
        "pre_roll_seconds": metadata.get("pre_roll_seconds", 0),
        "audio_seconds": audio_info.duration,
        "sample_rate": audio_info.samplerate,
    })
assert inputs, "No matching utterances found."
print(f"Selected {len(inputs)} clips; {sum(item['audio_seconds'] for item in inputs):.3f} seconds.")
print("Selection uses the saved original transcript, not the new model output.")
print("Audio sample rates:", sorted({item["sample_rate"] for item in inputs}))
display(HTML("<p>First input: " + escape(inputs[0]["id"]) + " — " + escape(inputs[0]["original_text"]) + "</p>"))


Selected 48 clips; 113.200 seconds.
Selection uses the saved original transcript, not the new model output.
Audio sample rates: [48000]


## Direct model test
Each clip is sent to the deployed model with the Subway hotword and Thai/English language guidance, without additional timestamp instructions. The built-in transcription prompt required by the model chat interface remains in use. Each clip gets a fresh session and is unloaded in `finally`, including errors. Results are saved after every clip.



In [3]:
OUTPUT_DIR = new_run_dir(ROOT / "outputs/greetings-1b1c0ac2-finetuned")
OUTPUT_DIR.mkdir(parents=True)
results = []
runtime: dict = {"clips": []}
session = raw_output = segments = None
started = perf_counter()
provenance = {
    "source_run": str(SOURCE_RUN), "keyword": KEYWORD,
    "model_id": settings["model_id"], "manifest": manifest,
    "ov_device": settings["ov_device"], "languages": settings["languages"],
    "hotwords": settings["hotwords"], "max_new_tokens": MAX_NEW_TOKENS,
    "allow_incomplete": True, "model_lifecycle": "Load and unload for every input clip.",
    "selection_count": len(inputs), "started_utc": datetime.now(timezone.utc).isoformat(),
    "input_scope": "Saved utterance WAVs, including their existing pre-roll; no VAD or collection pipeline.",
}
(OUTPUT_DIR / "inputs.json").write_text(json.dumps(inputs, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
try:
    for index, item in enumerate(inputs, 1):
        clip_runtime = {"id": item["id"]}
        clip_started = perf_counter()
        try:
            with OpenVinoSession(settings["model_id"], settings["model_dir"],
                                 ov_device=settings["ov_device"], audio_window_seconds=None) as session:
                assert session.processor is not None
                if index == 1:
                    print("Model input sample rate:", session.processor.feature_extractor.sampling_rate)
                inference_started = perf_counter()
                raw_output, segments = session.transcribe(
                    item["audio_path"], MAX_NEW_TOKENS, languages=settings["languages"], hotwords=settings["hotwords"], allow_incomplete=True)
                result = {**item, "new_text": " ".join(segment.text for segment in segments),
                          "raw_output": raw_output, "segments": [asdict(segment) for segment in segments],
                          "inference_seconds": perf_counter() - inference_started,
                          "generation_complete": all(session.runtime_metrics["openvino_batches"][-1]["generation_complete"]),
                          "error": None}
        except Exception as error:
            result = {**item, "new_text": "", "raw_output": "", "segments": [],
                      "inference_seconds": None, "generation_complete": False,
                      "error": f"{type(error).__name__}: {error}"}
        finally:
            if session is not None:
                clip_runtime.update(session.runtime_metrics)
            session = raw_output = segments = None
            release_model_memory(torch.device("cpu"))
            clip_runtime["workflow_seconds"] = perf_counter() - clip_started
            memory = clip_runtime.get("ov_memory_after_unload")
            clip_runtime["memory_released"] = memory is not None and all(value == 0 for value in memory.values())
            runtime["clips"].append(clip_runtime)
            if memory is not None:
                assert clip_runtime["memory_released"], clip_runtime
        results.append(result)
        (OUTPUT_DIR / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"{index}/{len(inputs)} {item['id']}: complete={result['generation_complete']} "
              f"memory_released={clip_runtime['memory_released']} "
              f"output={result['new_text'] or result['raw_output'] or result['error']}", flush=True)
finally:
    session = raw_output = segments = None
    release_model_memory(torch.device("cpu"))
    runtime["torch_cpu_allocated_after_unload"], _ = tensor_memory(torch.device("cpu"))
    if torch.xpu.is_available():
        release_model_memory(torch.device("xpu:0"))
        runtime["torch_xpu_allocated_after_unload"], runtime["torch_xpu_reserved_after_unload"] = tensor_memory(torch.device("xpu:0"))
    if torch.cuda.is_available():
        release_model_memory(torch.device("cuda:0"))
        runtime["torch_cuda_allocated_after_unload"], runtime["torch_cuda_reserved_after_unload"] = tensor_memory(torch.device("cuda:0"))
    runtime["workflow_seconds"] = perf_counter() - started
    runtime["completed_count"] = len(results)
    runtime["error_count"] = sum(item["error"] is not None for item in results)
    runtime["all_clips_memory_released"] = all(item["memory_released"] for item in runtime["clips"])
    (OUTPUT_DIR / "runtime.json").write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding="utf-8")

assert runtime["all_clips_memory_released"], runtime
assert all(value == 0 for key, value in runtime.items()
           if key.startswith("torch_") and key.endswith("after_unload")), runtime
print("Every clip released OpenVINO allocations:", runtime["all_clips_memory_released"])
print("Completed:", len(results), "Errors:", runtime["error_count"])
print("Saved results:", OUTPUT_DIR)




[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


OpenVINO execution: {'backend': 'openvino', 'ov_device': 'GPU', 'torch_allocator_scope': 'CPU preprocessing only; not OpenVINO GPU memory', 'audio_window_seconds': None, 'openvino_batches': [], 'openvino_batch_count': 0, 'metric_history_limit': 100, 'ov_version': '2026.4.1-22982-e213a147257-releases/2026/4', 'ov_device_name': 'Intel(R) Graphics (iGPU)', 'ov_memory_before_compile': {'cl_mem': 0, 'unknown': 0, 'usm_device': 0, 'usm_host': 0, 'usm_shared': 0}, 'encoder_execution_devices': ['GPU.0'], 'decoder_execution_devices': ['GPU.0']}
Model input sample rate: 16000
OpenVINO generated 1/128 steps
1/48 segment-000026/utterance-0001: complete=True memory_released=True output=สวัสดีค่ะ
OpenVINO execution: {'backend': 'openvino', 'ov_device': 'GPU', 'torch_allocator_scope': 'CPU preprocessing only; not OpenVINO GPU memory', 'audio_window_seconds': None, 'openvino_batches': [], 'openvino_batch_count': 0, 'metric_history_limit': 100, 'ov_version': '2026.4.1-22982-e213a147257-releases/2026/4'

KeyboardInterrupt: 

## Model output
Listen to each input and inspect the model-generated transcript and raw response. No transcript comparison or accuracy score is produced. Errors and token-limited outputs are marked explicitly.


In [ ]:
rows = []
for index, result in enumerate(results, 1):
    audio_bytes = Path(result["audio_path"]).read_bytes()
    encoded_audio = base64.b64encode(audio_bytes).decode("ascii")
    status = (
        "Error" if result["error"]
        else "Token limit (partial)" if not result["generation_complete"]
        else "Output parsed" if result["segments"]
        else "Output unparsed"
    )
    rows.append(
        f"<tr><td>{index}<br>{escape(result['id'])}<br>"
        f"{result['audio_seconds']:.3f} s / {result['sample_rate']} Hz<br>"
        f"<audio controls preload=\"none\" src=\"data:audio/wav;base64,{encoded_audio}\"></audio></td>"
        f"<td>{escape(result['new_text'] or result['raw_output'] or result['error'])}</td>"
        f"<td>{status}<br>Inference: {result['inference_seconds']} s<br>"
        f"{escape(result['error'] or '')}</td></tr>"
    )

comparison_html = (
    "<style>.model-test{border-collapse:collapse;width:100%;table-layout:fixed}"
    ".model-test th,.model-test td{border:1px solid #aaa;padding:10px;"
    "vertical-align:top;overflow-wrap:anywhere}"
    ".model-test audio{width:100%;min-width:160px}</style>"
    f"<p><b>{len(results)} clips</b> | Errors: {runtime['error_count']}</p>"
    "<table class=\"model-test\"><thead><tr><th>Input audio</th><th>Model output</th>"
    "<th>Status / inference time</th></tr></thead><tbody>"
    + "".join(rows)
    + "</tbody></table>"
)

display(HTML(comparison_html))
(OUTPUT_DIR / "model_test.html").write_text(
    "<!doctype html><html lang=\"en\"><meta charset=\"utf-8\">"
    "<title>Model test: 1b1c0ac2</title>"
    + comparison_html
    + "</html>",
    encoding="utf-8",
)
assert len(results) == len(inputs), "Some inputs were not attempted; inspect saved results."

Input audio,Model output,Status / inference time
1segment-000026/utterance-00011.500 s / 48000 Hz,สวัสดีค่ะ,Output parsedInference: 2.487695919999169 s
2segment-000026/utterance-00021.340 s / 48000 Hz,[0.53][S01][0.71][0.56][S02][0.75],Output unparsedInference: 2.602564197999527 s
3segment-000056/utterance-00021.880 s / 48000 Hz,ฮัลโหล สวัสดีค่ะ,Output parsedInference: 2.459771231000559 s
4segment-000227/utterance-00012.180 s / 48000 Hz,Subway สวัสดีค่ะ,Output parsedInference: 2.1099286310000025 s
5segment-000229/utterance-00082.350 s / 48000 Hz,Subway สวัสดีค่ะ,Output parsedInference: 2.2792441830006283 s
6segment-000232/utterance-00063.800 s / 48000 Hz,สวัสดีค่ะ เลิกดูเมนูก่อนได้นะคะ,Output parsedInference: 2.800037164000969 s
7segment-000235/utterance-00012.500 s / 48000 Hz,"Rabes, สวัสดีค่ะ",Output parsedInference: 2.6071652699993138 s
8segment-000237/utterance-00062.280 s / 48000 Hz,สวัสดีค่ะ,Output parsedInference: 2.2707859599995572 s
9segment-000243/utterance-00011.940 s / 48000 Hz,Subway สวัสดีค่ะ,Output parsedInference: 2.401863746001254 s
10segment-000244/utterance-00012.360 s / 48000 Hz,Subway สวัสดีค่ะ,Output parsedInference: 2.3465203719988494 s
